# Data Preparation Workflows for Prototype and Final Runs with data_preparation.py

This implementation notebook documents all data preparation steps, from sampling raw FASTA files for each genomic feature, to constructing balanced training and testing datasets for the final classification implementation. Because this study investigates how varying the Markov model order k influences classification accuracy, it was essential that the underlying sequences be unambiguous in their biological identity. Any uncertainty in feature identity would confound the interpretation of model performance, making it impossible to attribute changes in accuracy solely to the Markov order. Therefore, all raw data were obtained from UCSC Genome Browser/GENCODE's v19 annotation release, aligned to the H. sapiens GRCh37.p13 genome assembly (freeze year 2013). This collection was deliberately chosen to ensure that every dataset was derived from high‑quality, experimentally validated annotations with long, well‑characterized usage histories.

The promoter sequences were obtained from the EPDnew v006 track as hosted within the UCSC Genome Browser for the hg19 assembly. Although EPDnew is curated by the Swiss Institute of Bioinformatics, the UCSC‑hosted version ensures that all promoter coordinates and extracted sequences are aligned to the same hg19/GRCh37 reference genome used for the rest of the dataset. Exons and introns were extracted directly from the GENCODE v19 annotation and the corresponding GRCh37.p13 reference genome, which is the Genome Reference Consortium assembly that UCSC designates as hg19.

GENCODE v19 is the final GENCODE release for GRCh37, and no exon or intron FASTA files were ever distributed for this assembly. Furthermore, UCSC deprecated hg19 feature‑level FASTA export around 2023, leaving me with no options for directly accessing the FASTA files for my intron and exon classes from an online database. Instead, I downloaded the full genome FASTA and reconstructed exon and introns with gffread on my local terminal.

By this method, exons are identified from the GTF/GFF v19 annotation, because they are explicitly listed as exon features with chromosome, start, end, and strand coordinates. Introns, on the other hand, are not listed in the GTF. Therefore, gffread infers them by taking the gaps between consecutive exons from the same transcript, respecting the transcript’s strand. It then extracts those genomic intervals as intron sequences. The resulting exon and intron sequences are considered highly reliable and structurally accurate representations of the underlying gene models.

Finally, repeats dataset was extracted from UCSC's RepeatMasker database. Below, I have documented all preprocessing steps in both the prototype and final implementations of the model. For the prototype run, only promoters and repeats were included, as this allowed me to demonstrate full proficiency of the model pipeline while still being able to spot issues and opportunities to improve the workflow for the final run.


### Package and Module Imports

In [1]:
import os
from Bio import SeqIO
from data_preparation import load_fasta, count_seqs, filter_by_length, sample_fasta, rewrite_headers, remove_sampled_from_dict, sample_negative_candidates, shuffle_records_dinuc, infer_dataclass, compute_class_stats, write_preliminary_csv

### Set Working Directory to Root Directory and Add src/ and notebook/ to Path

In [8]:
import os
import sys

cwd = os.getcwd()
# Ensure working dir is root, not src/ or notebooks/
if cwd.endswith("src") or cwd.endswith("notebooks"):
    os.chdir("..")

print("Working directory:", os.getcwd())

# Add src/ to Python path
sys.path.insert(0, "src")
sys.path.insert(0, "notebook")

Working directory: /Users/biotechiestefnie/Desktop/Algorithms_Final_Project


## **Data Preprocessing Pipeline for Prototype Run**

For the Prototype Run, positive and negative datasets containing 300 sequences each were generated for all structural classes from the raw FASTA files. A dedicated prototype/ subdirectory was created inside the top‑level data/ directory to store the eight resulting files (four genomic classes × positive/negative). Each file is named using the convention class_label.fa, where class denotes the genomic feature and label indicates positive or negative membership.

For demonstration purposes, only the promoter and repeat datasets are used in the prototype pipeline to validate end‑to‑end functionality: data loading, model training, model testing, result generation, evaluation, plotting, and writing outputs to file. In keeping with the nature of a prototype, no biological or statistical conclusions are drawn from these results. Instead, outputs are inspected only to confirm that they resemble expected behavior at a high level—for example, raw log‑likelihood scores should be negative and approximately proportional to sequence length (on the order of ~1.5 × L for transition/emission probabilities near 0.25).

In [4]:
"""
 Prototype data preprocessing calls:
    - infer class type from filepath
    - load raw FASTA into a dictionary {seq_id: SeqRecord}
    - count raw sequences for each class
    - filter exons/introns for valid lengths (promoters/repeats unchanged)
    - sample positive sequences (k=300)
    - rewrite headers for positive sequences
    - write positive FASTA file
    - remove sampled positives from dictionary
    - sample negative candidates from remaining sequences
    - apply dinucleotide shuffling to destroy biological signal
    - rewrite headers for negative sequences
    - write negative FASTA file
    - compute preliminary statistics for all positive and negative datasets
    - write summary CSV of prototype dataset characteristics
"""

raw_dir = "data/raw"
out_dir = "data/prototype"

# make sure the directory really exists from this notebook's cwd
os.makedirs(out_dir, exist_ok=True)
print("Prototype out_dir:", os.path.abspath(out_dir))
print("Contents before run:", os.listdir(out_dir))

fasta_files = [
    "promoters_raw.fa",
    "exons_raw.fa",
    "introns_raw.fa",
    "repeats_raw.fa"
]

for fname in fasta_files:
    inpath = os.path.join(raw_dir, fname)

    feature, seq_dict = load_fasta(inpath)   # uses infer_dataclass
    # keep feature exactly as your helpers expect (lowercase)
    feature = feature.lower()

    print(f"\nProcessing {feature} from {os.path.abspath(inpath)}")
    count_seqs(seq_dict, feature)

    # POSITIVE
    pos_out = os.path.join(out_dir, f"{feature}_positive.fa")
    sampled_pos = sample_fasta(seq_dict, k=300)
    clean_pos = rewrite_headers(sampled_pos, f"{feature}_positive")
    n_pos = SeqIO.write(clean_pos, pos_out, "fasta")
    print(f"  wrote {n_pos} positives → {os.path.abspath(pos_out)}",
          "| exists:", os.path.exists(pos_out))

    # remove sampled from original dict (note: rewrite_headers mutates IDs in place)
    seq_dict_remaining = remove_sampled_from_dict(seq_dict, sampled_pos)
    print("  remaining after positive removal:", len(seq_dict_remaining))

    # NEGATIVE
    neg_candidates = sample_negative_candidates(seq_dict_remaining, k=300)
    neg_records = shuffle_records_dinuc(neg_candidates)

    neg_out = os.path.join(out_dir, f"{feature}_negative.fa")
    clean_neg = rewrite_headers(neg_records, f"{feature}_negative")
    n_neg = SeqIO.write(clean_neg, neg_out, "fasta")
    print(f"  wrote {n_neg} negatives → {os.path.abspath(neg_out)}",
          "| exists:", os.path.exists(neg_out))

    print(f"Finished {feature}")

# preliminary stats
prelim_rows = []
prelim_csv = os.path.join(out_dir, "preliminary_characteristics.csv")

for fname in fasta_files:
    feature = infer_dataclass(fname).lower()
    source_file = fname

    pos_path = os.path.join(out_dir, f"{feature}_positive.fa")
    neg_path = os.path.join(out_dir, f"{feature}_negative.fa")

    print(f"\nStats for {feature}:")
    print("  pos_path:", os.path.abspath(pos_path), "exists:", os.path.exists(pos_path))
    print("  neg_path:", os.path.abspath(neg_path), "exists:", os.path.exists(neg_path))

    prelim_rows.append(
        compute_class_stats(pos_path, feature, "positive", source_file)
    )
    prelim_rows.append(
        compute_class_stats(neg_path, feature, "negative", source_file)
    )

write_preliminary_csv(prelim_rows, prelim_csv)
print(f"\nWrote preliminary characteristics csv → {os.path.abspath(prelim_csv)}")
print("Contents after run:", os.listdir(out_dir))



Prototype out_dir: /Users/biotechiestefnie/Desktop/Algorithms_Final_Project/data/prototype
Contents before run: ['exons_positive.fa', 'introns_negative.fa', 'promoters_positive.fa', 'promoters_negative.fa', 'introns_positive.fa', 'exons_negative.fa', 'repeats_negative.fa', 'preliminary_characteristics.csv', 'repeats_positive.fa']

Processing promoters from /Users/biotechiestefnie/Desktop/Algorithms_Final_Project/data/raw/promoters_raw.fa
Total sequences in promoters raw file: 1796
  wrote 300 positives → /Users/biotechiestefnie/Desktop/Algorithms_Final_Project/data/prototype/promoters_positive.fa | exists: True
  remaining after positive removal: 1796
  wrote 300 negatives → /Users/biotechiestefnie/Desktop/Algorithms_Final_Project/data/prototype/promoters_negative.fa | exists: True
Finished promoters

Processing exons from /Users/biotechiestefnie/Desktop/Algorithms_Final_Project/data/raw/exons_raw.fa
Total sequences in exons raw file: 104142
  wrote 300 positives → /Users/biotechiestef

The output above confirms that positive and negative datasets for all four structural classes were successfully generated and written to the data/prototype/ directory. While promoter sequences were standardized to a uniform length of 501bp during extraction from EPDnew, the lengths of sequences in the other classes were not known in advance. To address this, a summary CSV file was also produced to capture the minimum, maximum, mean, and distributional characteristics of sequence lengths for each class. These statistics provide essential context for understanding dataset composition and for anticipating how sequence length might influence model behavior in subsequent classification experiments.

## **Data Preprocessing Pipeline for Final Implementation: Training and Testing Sets**

During the prototype run, I observed that the raw log‑likelihood scores for the promoter and repeat datasets showed substantial overlap between the positive and negative classes, which prevented the model from making reliable classifications. To address this, I applied per‑base normalization to remove the strong length‑dependent bias inherent in cumulative log‑likelihoods. This correction greatly improved class separability; however, sequence length continued to influence model behavior through variance rather than bias.

Long sequences contain many transitions, k‑mers, and emission events, so their per‑base log‑likelihood represents an average over hundreds or thousands of observations. As a result, these values become extremely stable, with very little fluctuation across sequences of similar length. This “over‑stability” can produce artificially high classification confidence and can distort the estimated transition probabilities during training. In contrast, very short sequences contain only a small number of transitions and k‑mers, which leads to noisy, high‑variance per‑base log‑likelihoods and reduced classification reliability.

### Extract Sequences from Raw Datafiles and Run Preliminary Statistics to Determine Length Thresholds

In [5]:
# Import packages
import os
from Bio import SeqIO

# Directories
raw_dir    = "data/raw"
final_dir  = "data/final"
master_dir = os.path.join(final_dir, "master")
os.makedirs(master_dir, exist_ok=True)

# Output CSV for preliminary stats
prelim_csv = os.path.join(final_dir, "preliminary_stats.csv")

# Classes to process
classes = ["promoters", "exons", "introns", "repeats"]

# Collect rows for csv
prelim_rows = []

for feature in classes:
    print(f"\nProcessing {feature}")

    # Raw input path
    inpath = os.path.join(raw_dir, f"{feature}_raw.fa")

    # Load raw FASTA
    feature_name, seq_dict = load_fasta(inpath)

    # Sample 1700 sequences
    sampled = sample_fasta(seq_dict, k=1700)

    # Rewrite headers
    clean = rewrite_headers(sampled, f"{feature_name}_all_final")

    # Write master FASTA
    master_out = os.path.join(master_dir, f"{feature_name}_all_final.fa")
    SeqIO.write(clean, master_out, "fasta")
    print(f"  → Wrote {master_out}")

    # Compute preliminary statistics for this class
    stats_row = compute_class_stats(
        fasta_path=master_out,
        feature=feature_name,
        label="final",
        source_file=f"{feature}_raw.fa"
    )
    prelim_rows.append(stats_row)

    print(stats_row)

# Write the preliminary stats CSV
write_preliminary_csv(prelim_rows, prelim_csv)
print(f"\nWrote preliminary statistics CSV → {prelim_csv}")



Processing promoters
  → Wrote data/final/master/promoters_all_final.fa
{'feature': 'promoters', 'label': 'final', 'total_seqs': 1700, 'min_len': 501, 'max_len': 501, 'mean_len': 501.0, 'median_len': 501.0, 'standard_dev': 0.0, 'iq_range': 0.0, 'coeff_var': 0.0, '% A': 18.08, '% T': 18.97, '% G': 32.64, '% C': 30.16, '% GC_cont': 62.8, 'source_file': 'promoters_raw.fa'}

Processing exons
  → Wrote data/final/master/exons_all_final.fa
{'feature': 'exons', 'label': 'final', 'total_seqs': 1700, 'min_len': 3, 'max_len': 14586, 'mean_len': 1103.89, 'median_len': 675.0, 'standard_dev': 1256.92, 'iq_range': 1052.0, 'coeff_var': 1.139, '% A': 25.18, '% T': 21.72, '% G': 27.0, '% C': 26.11, '% GC_cont': 53.11, 'source_file': 'exons_raw.fa'}

Processing introns
  → Wrote data/final/master/introns_all_final.fa
{'feature': 'introns', 'label': 'final', 'total_seqs': 1700, 'min_len': 2, 'max_len': 177981, 'mean_len': 6323.86, 'median_len': 1568.0, 'standard_dev': 15779.87, 'iq_range': 4198.5, 'coef

The promoter class behaves exactly as expected: all 1700 sequences are uniformly 501bp, reflecting the fixed extraction window around the transcription start site. Their consistent GC content (~63%) and complete lack of length variability make promoters a useful internal control for the dataset.

Exons show broad heterogeneity, ranging from 10bp to 15.6kb. The median length (721bp) sits well below the mean (1144bp), indicating a right‑skewed distribution with both micro‑exons and unusually long exons represented. Although biologically plausible, this wide spread necessitates moderate length filtering to retain the central, representative portion of the distribution.

Introns exhibit the widest range of any class, spanning from 2bp to over 442kb. The median (1547bp) is substantially lower than the mean (6418bp), reflecting a heavy right tail of very long introns. These extremes sit far outside the central distribution and require more aggressive upper‑bound trimming than the other classes.

Repeats display the highest relative variability, with a very small median length (55bp) but a long tail extending beyond 55kb. This reflects the mixture of simple repeats, SINEs, LINEs, and LTR fragments. The combination of ultra‑short elements and large retrotransposon fragments produces a highly dispersed distribution that would greatly benefit from both lower‑ and upper‑bound filtering.

Taken together, these distributions show that each variable‑length class contains extreme outliers that must be trimmed to preserve the representative core of each feature type. Based on the observed ranges, I will apply class‑specific thresholds: exons were restricted to 150–5000bp, introns to 500–10000bp, and repeats to 50–8000bp. These bounds retain the biologically typical sequences while excluding the tails that fall far outside each class’s central distribution.

### Filter Exons, Introns, and Repeats by Length, Split into 80/20 Training/Testing Datasets, Randomize and Shuffle Testing Set and Write to Files for Final Run

In [6]:
import os, random
from Bio import SeqIO

THRESHOLDS = {
    "exons":   (150, 5000),
    "introns": (500, 10000),
    "repeats": (50, 8000)
}

filter_classes = ["exons", "introns", "repeats"]
all_classes    = ["promoters", "exons", "introns", "repeats"]

master_dir = "data/final/master"
train_out  = "data/final/train/train.fa"
test_out   = "data/final/test/test.fa"


def filter_by_length(records, feature):
    min_len, max_len = THRESHOLDS[feature]
    return [r for r in records if min_len <= len(r.seq) <= max_len]

train_all = []
test_all  = []

for feature in all_classes:
    inpath = f"{master_dir}/{feature}_all_final.fa"
    records = list(SeqIO.parse(inpath, "fasta"))

    if feature in filter_classes:
        records = filter_by_length(records, feature)

    random.seed(43)
    random.shuffle(records)

    split_idx = int(0.8 * len(records))
    train_all.extend(records[:split_idx])
    test_all.extend(records[split_idx:])

# write sequences to train folder as one file of 4 classes
SeqIO.write(train_all, train_out, "fasta")

# shuffle seqs and combine into testing file in test folder
random.seed(43)
random.shuffle(test_all)
SeqIO.write(test_all, test_out, "fasta")

print("TRAIN:", train_out, "→", len(train_all), "records")
print("TEST:",  test_out,  "→", len(test_all),  "records")




TRAIN: data/final/train/train.fa → 4165 records
TEST: data/final/test/test.fa → 1043 records


The raw datafiles created at the beginning of the project were sampled to extract 1,700 sequences for each class. This sample size was determined by the smallest available class—promoters (1,796 sequences)—ensuring that all classes could contribute equally to the training and testing subsets. After extracting these sequences into the master directory, I performed a preliminary statistical analysis to evaluate the length distributions of each class.

Although all sequences were experimentally validated and correctly annotated, I applied length‑based filtering to the intron, exon, and repeat classes. This was not intended to eliminate non-class sequences; rather, it was necessary to prevent length‑induced distortion in the Markov Chain models. Both effects can cause the classifier to rely on sequence length rather than true class‑specific patterns. The thresholds were therefore chosen to remove pathological extremes while preserving the vast majority of meaningful signal.

After filtering, the promoter, exon, intron, and repeat datasets were combined and split into 80/20 training and testing sets, resulting in a training FASTA containing 4,165 sequences and a testing FASTA containing 1,043 sequences. The test sequences were shuffled to randomize class order. At this stage, both training and testing sequences retain their class labels; however, these labels will be removed prior to model exposure during the final run so that classification is driven solely by the learned sequence patterns. The following cell will display the top five sequences in the training and testing file for confirmation that they were written correctly to file.

In [ ]:
from Bio import SeqIO

fasta_path = "data/final/train/train.fa"

for i, record in enumerate(SeqIO.parse(fasta_path, "fasta"), start=1):
    print(f">{record.id}")
    print(record.seq)
    if i == 5:
        break
